# LangChain Agent — OpenAI + Ollama

Same logic as `app.py`, with the Ollama section updated for local-server and model checks.


## Imports


In [1]:
import os
import certifi
import requests
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.chat_models.ollama import ChatOllama
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.agents import create_react_agent
from langchain.agents.agent import AgentExecutor


## 1. LOAD ENVIRONMENT VARIABLES


In [ ]:
os.environ["SSL_CERT_FILE"] = certifi.where()

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

env_candidates = [
    BASE_DIR / ".env",
    BASE_DIR.parent / ".env",
    BASE_DIR / "research" / ".env",
]

env_path = next((p for p in env_candidates if p.exists()), env_candidates[0])
load_dotenv(dotenv_path=env_path, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

# auto   -> try OpenAI first, then fall back to Ollama
# openai -> OpenAI only
# ollama -> local Ollama only
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "auto").lower()

# Ollama settings
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

print("ENV file:", env_path)
print("ENV exists:", env_path.exists())
print("OpenAI key loaded:", bool(OPENAI_API_KEY))
print("Tavily key loaded:", bool(TAVILY_API_KEY))
print("Weatherstack key loaded:", bool(WEATHERSTACK_API_KEY))
print("LLM provider:", LLM_PROVIDER)
print("Ollama model:", OLLAMA_MODEL)


ENV file: c:\langchain.agent\research\.env
ENV exists: True
OpenAI key loaded: True
Tavily key loaded: True
Weatherstack key loaded: True
LLM provider: auto
Ollama model: llama3.2:3b


## 2. TOOLS


In [4]:
@tool
def get_weather_data(city: str) -> str:
    """Fetch current weather information for a city."""

    url = (
        f"https://api.weatherstack.com/current"
        f"?access_key={WEATHERSTACK_API_KEY}"
        f"&query={city}"
    )

    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as error:
        return f"Weather request failed: {error}"

    if "error" in data:
        return f"Weather API error: {data['error']}"

    location = data["location"]
    current = data["current"]

    return (
        f"City: {location['name']}\n"
        f"Country: {location['country']}\n"
        f"Temperature: {current['temperature']}°C\n"
        f"Feels Like: {current['feelslike']}°C\n"
        f"Weather: {current['weather_descriptions'][0]}\n"
        f"Humidity: {current['humidity']}%\n"
        f"Wind Speed: {current['wind_speed']} km/h"
    )


search_tool = TavilySearchResults(
    max_results=2,
    tavily_api_key=TAVILY_API_KEY
)

tools = [search_tool, get_weather_data]


## 3. OPENAI MODEL


In [5]:
openai_llm = None

if OPENAI_API_KEY:
    openai_llm = ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0,
        api_key=OPENAI_API_KEY
    )


## 4. UPDATED OLLAMA SECTION


In [6]:
def check_ollama():
    """Check that Ollama is running and the selected model is installed."""

    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as error:
        raise RuntimeError(
            "\nOllama is not reachable.\n"
            "Open the Ollama app or run:\n"
            "    ollama serve\n\n"
            f"Original error: {error}"
        )

    installed_models = [m.get("name", "") for m in data.get("models", [])]

    model_found = any(
        name == OLLAMA_MODEL or name.startswith(f"{OLLAMA_MODEL}:")
        for name in installed_models
    )

    if not model_found:
        raise RuntimeError(
            f"\nOllama is running, but '{OLLAMA_MODEL}' is not installed.\n"
            "Install it with:\n"
            f"    ollama pull {OLLAMA_MODEL}\n"
        )

    print("Ollama status: Ready")


def get_ollama_llm():
    """Create the local Ollama model only when it is needed."""

    check_ollama()

    return ChatOllama(
        model=OLLAMA_MODEL,
        temperature=0,
        base_url=OLLAMA_BASE_URL
    )


## 5. REACT PROMPT


In [7]:
prompt = hub.pull("hwchase17/react")


c:\Users\DELL\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


## 6. CREATE AGENT EXECUTOR


In [8]:
def create_executor(llm):
    agent = create_react_agent(
        llm=llm,
        tools=tools,
        prompt=prompt
    )

    return AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,
        handle_parsing_errors=True
    )


## 7. RUN WITH OPENAI / OLLAMA / AUTOMATIC FALLBACK


In [9]:
def run_agent(question: str):

    if LLM_PROVIDER == "openai":
        if openai_llm is None:
            raise ValueError("OPENAI_API_KEY was not found in the .env file.")

        print("\nUsing OpenAI...\n")
        executor = create_executor(openai_llm)
        return executor.invoke({"input": question})

    if LLM_PROVIDER == "ollama":
        print(f"\nUsing local Ollama model: {OLLAMA_MODEL}\n")
        ollama_llm = get_ollama_llm()
        executor = create_executor(ollama_llm)
        return executor.invoke({"input": question})

    if LLM_PROVIDER == "auto":
        if openai_llm is not None:
            try:
                print("\nTrying OpenAI first...\n")
                executor = create_executor(openai_llm)
                return executor.invoke({"input": question})
            except Exception as error:
                print("\nOpenAI could not complete the request.")
                print("Reason:", error)
                print(f"\nAutomatically switching to Ollama: {OLLAMA_MODEL}\n")
        else:
            print(f"\nNo OpenAI API key found. Using Ollama: {OLLAMA_MODEL}\n")

        ollama_llm = get_ollama_llm()
        executor = create_executor(ollama_llm)
        return executor.invoke({"input": question})

    raise ValueError("LLM_PROVIDER must be 'auto', 'openai', or 'ollama'.")


## 8. RUN THE AGENT


In [10]:
question = "Find the capital of Pakistan and then find its current weather."

response = run_agent(question)

print("\nFINAL ANSWER:")
print(response["output"])



Trying OpenAI first...



> Entering new AgentExecutor chain...

OpenAI could not complete the request.
Reason: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

Automatically switching to Ollama: llama3.2:3b



RuntimeError: 
Ollama is running, but 'llama3.2:3b' is not installed.
Install it with:
    ollama pull llama3.2:3b
